# Confidence Intervals and Bootstrap

## Overview

A 95% CI is a procedure that, over repeated sampling, contains the true parameter 95% of the time. It does NOT mean "95% probability the true value is in this specific interval."

Bootstrap CIs require no distributional assumptions and work for any statistic:
1. Resample with replacement B times
2. Compute the statistic on each resample
3. Take percentiles of the bootstrap distribution

Use BCa (bias-corrected accelerated) for skewed statistics. B >= 2000 for reliable results.

---

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from scipy.stats import bootstrap as sci_boot
from statsmodels.stats.proportion import proportion_confint
rng = np.random.default_rng(42)
n = 80
richness = rng.gamma(4, 4.5, n)
print(f"n={n}, mean={richness.mean():.2f}, median={np.median(richness):.2f}")

---
## Parametric CIs

In [ ]:
# CI for mean (t-distribution)
ci_t = stats.t.interval(0.95, df=n-1, loc=richness.mean(), scale=stats.sem(richness))
print(f"95% CI for mean (t): [{ci_t[0]:.2f}, {ci_t[1]:.2f}]")
# Wilson CI for proportions -- better than normal approx near 0/1
k = 23
lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
print(f"Wilson CI for {k}/{n}={k/n:.3f}: [{lo:.3f}, {hi:.3f}]")

---
## Bootstrap Percentile CIs

In [ ]:
B = 5000
boots_mean = [rng.choice(richness, n, replace=True).mean() for _ in range(B)]
boots_med  = [np.median(rng.choice(richness, n, replace=True)) for _ in range(B)]
ci_mean = np.percentile(boots_mean, [2.5, 97.5])
ci_med  = np.percentile(boots_med,  [2.5, 97.5])
print(f"Bootstrap CI for mean:   [{ci_mean[0]:.2f}, {ci_mean[1]:.2f}]")
print(f"Bootstrap CI for median: [{ci_med[0]:.2f},  {ci_med[1]:.2f}]")
# BCa via scipy -- preferred for skewed statistics
bca = sci_boot((richness,), np.mean, confidence_level=0.95,
               n_resamples=B, method="BCa", random_state=42)
print(f"BCa CI for mean: [{bca.confidence_interval.low:.2f}, {bca.confidence_interval.high:.2f}]")

---
## Visualising Bootstrap Distributions

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4))
for ax, dist, obs, ci, lbl in zip(
    axes, [boots_mean, boots_med],
    [richness.mean(), np.median(richness)],
    [ci_mean, ci_med], ["Mean","Median"]):
    ax.hist(dist, bins=50, color="steelblue", alpha=0.7, density=True, edgecolor="white")
    ax.axvline(obs, color="navy", lw=2, label=f"Observed {lbl}={obs:.2f}")
    ax.axvline(ci[0], color="#e74c3c", lw=1.5, linestyle="--", label="95% CI")
    ax.axvline(ci[1], color="#e74c3c", lw=1.5, linestyle="--")
    ax.set_title(f"Bootstrap {lbl}")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

---
## CI Width vs Sample Size

In [ ]:
ns = [20,40,80,160,320]
widths = []
for n_s in ns:
    s = rng.gamma(4,4.5,n_s)
    ci = stats.t.interval(0.95,df=n_s-1,loc=s.mean(),scale=stats.sem(s))
    widths.append(ci[1]-ci[0])
plt.figure(figsize=(7,4))
plt.plot(ns, widths, "o-", color="steelblue", lw=2)
plt.xlabel("Sample size"); plt.ylabel("CI width")
plt.title("CI width ~ 1/sqrt(n): quadrupling n halves the width")
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Misinterpreting a CI as a probability statement**  
Once data are collected the CI either contains the true value or it does not. The 95% refers to the long-run procedure, not to any specific interval.

**2. Using normal-approximation CI for proportions near 0 or 1**  
The standard formula can produce bounds outside [0,1]. Use the Wilson interval via `proportion_confint(method="wilson")`.

**3. Using B=100 bootstrap resamples**  
With B=100, CIs vary substantially between runs. Use B >= 1000 for percentile and B >= 2000 for BCa.

**4. Using percentile bootstrap for skewed statistics**  
For skewed or biased statistics the percentile CI is itself biased. Use BCa via `scipy.stats.bootstrap(method="BCa")`.

**5. Not stating the CI method and confidence level**  
Always report the confidence level, method (t, Wilson, bootstrap percentile, BCa), and sample size alongside any CI.
---
*python_methods_library - Samantha McGarrigle*